# 04 - Baseline link prediction

> **Purpose.** This notebook is a demonstration of the code base. It shows how to load the dataset, what the graph looks like once the shortcut controls are applied, and how a HeteroSAGE baseline is trained and evaluated with the same code as `training/train.py`. It is meant for understanding the code and the data, not for reproducing the reported results. Those come from `training/train.py` and `training/train_seeds.py` (full schedule, 5 seeds; see the README). The defaults below follow `train.py`, with a shorter training schedule.

**Task.** Given a biochemical reaction (`Conversion` node), rank the candidate proteins and identify the one that catalyses it: link prediction on `(Protein, catalyzes, Interaction)`. Validation and test edges come from organisms that are held out entirely from training (the taxa holdout).

**Metrics.**

- **P-H@K** (Protein Hits at K), the primary metric: for each validation or test reaction, all embedded proteins are ranked against it, and the reaction is a hit if the true catalyst is in the top K. A random ranking gives K divided by the pool size (about 0.59% at K=50 for the 8,445 embedded proteins).
- **CP-AUC / CP-AP**, secondary diagnostics: AUC and average precision of a true (Protein, Reaction) pair against the same reaction paired with a random protein.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
if not (ROOT / "training").exists() and (ROOT.parent / "training").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "training"))

from build_clean_ctx import load_ctx_for_run
from metrics import evaluate_cp_auc, evaluate_random_neg_auc, protein_hits_at_k
from models import build_model
from utils import collect_all_pairs, sample_training_negatives, set_seed

DATA_DIR = ROOT / "data"   # downloaded from Zenodo on first use

# Defaults of training/train.py, except EPOCHS (train.py runs 200)
GNN_NAME     = "sage"        # "residual_jumping_sage" is the HeteroSAGE+Res&Jump variant
HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 1e-4
WEIGHT_DECAY = 1e-5
EPOCHS       = 50
NEG_K        = 5             # (Protein, random Conversion) negatives per positive
NEG_K_CP     = 5             # (random Protein, Conversion) negatives per positive
SEED         = 42
USE_EMBEDDINGS = True        # False: random features, structural baseline

set_seed(SEED)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Load the graph and the taxa split

`load_ctx_for_run()` builds a `GraphContext` (in `training/dataset.py`) from `heterodata.pt`, `nodes.tsv` and `splits_taxa.pt`, downloading them from Zenodo if they are missing, and then applies the cleaning of `build_clean_ctx.py`. The settings below are the defaults of `train.py`.

Four choices remove structural shortcuts from the message-passing graph:

| Setting | Default | What it removes |
|---|---|---|
| `keep_pathways` | `False` | `(X, is_part_of, Pathway)` edges. A true (Protein, Conversion) pair almost always shares a Pathway and a random pair almost never does, so pathway co-membership alone gives AUC above 0.98. |
| `keep_catalyzed_by` | `False` | The reverse edge `(Interaction, catalyzed_by, Protein)`. It creates a 2-hop path P → C → P that lets the model memorise training pairs. |
| `disjoint_train_ratio` | `0.2` | The 1-hop shortcut (below). |
| `ec_features` | `False` | EC-number features on `Conversion` nodes. They give reaction-class identity directly and produce a large val-to-test gap, so they are only used as an ablation. |

**Disjoint supervision.** If every training positive (P, C) is also a message-passing edge, then after one `SAGEConv` layer z_P aggregates from C and z_C from P, and their dot product is high because the two nodes have seen each other, not because the model learned enzyme-substrate specificity. With `disjoint_train_ratio=0.2`, a random 20% of the training positives are used only as supervision and are never message-passing edges, so the model has to score them from features and neighbourhoods. Validation and test use the same 80% message-passing edges, so no validation or test edge is ever in the graph.

In [ ]:
HP = dict(
    seed=SEED, split_type="taxa", data_type="clean",
    keep_pathways=False, keep_catalyzed_by=False, disjoint_train_ratio=0.2, ec_features=False,
    remove_metabolite_organism_edges=True, bidirectional=True,
    download=True, print_dataset_summary=True,
)
ctx = load_ctx_for_run(HP, DATA_DIR)

### 1.1 What the context contains

In [ ]:
print("Node types (input feature dimension, rows with an all-zero feature vector):")
for nt, x in ctx.train_data.x_dict.items():
    n_zero = int((x.abs().sum(dim=1) == 0).sum())
    print(f"  {nt:<12} n={x.shape[0]:>7,}  dim={x.shape[1]:>5}  zero rows={n_zero:>6,}")

print("\nPositive target edges per split (taxa holdout):")
for name, ei in ctx.split_ei.items():
    print(f"  {name:<5} {ei.shape[1]:>6,}")

pool = int(ctx.embedded_protein_mask.sum())
print(f"\nRanking pool: {pool:,} proteins with an ESM-C embedding")
print(f"Random P-H@50 baseline: {50 / pool:.2%}")

## 2. Model

`build_model()` (in `training/models.py`) returns the GNN and the decoder in one module. `sage` is a `HeteroConv` of one `SAGEConv` per edge type with sum aggregation across edge types, followed by a per-layer `LayerNorm`; the decoder is a dot product between the Protein and Interaction embeddings. `SAGEConv(-1, -1)` infers the input dimension of each edge type on the first forward pass, which lets the same code handle the mixed feature dimensions (960 for Protein, 1,024 for Metabolite, 3,072 for Conversion).

The ReLU and dropout are applied between layers but not after the last one: the final embeddings feed a dot product, and clipping them at zero would make every dimension non-negative, so a wrong pair's score could not be pushed down. Without any non-linearity, two stacked affine layers would collapse into one.

Other architectures are selected with `gnn_name`: `residual_sage`, `residual_jumping_sage`, `hgt`, `gat`, `rgcn` and `mix`.

In [ ]:
model = build_model(
    ctx, gnn_name=GNN_NAME, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    decoder="dot", dropout=DROPOUT, random_seed=SEED, use_norm=True,
)

if not USE_EMBEDDINGS:
    # Structural baseline: same topology, random features. The ranking pool is unchanged.
    torch.manual_seed(SEED)
    for split in (ctx.train_data, ctx.val_data, ctx.test_data):
        for nt in split.node_types:
            dim = split[nt].x.shape[-1]
            split[nt].x = torch.randn(split[nt].num_nodes, dim, device=ctx.device)

## 3. Training

Each epoch takes the supervision positives of the training split and adds two kinds of negatives, drawn by `sample_training_negatives()` in `training/utils.py`:

- `(P, C_random)`: the same protein with a random reaction, `NEG_K` per positive, which trains the reaction-ranking direction.
- `(P_random, C)`: the same reaction with a random protein, `NEG_K_CP` per positive, which trains the protein-ranking direction that P-H@K evaluates.

Known true pairs are excluded from the negatives. The loss is binary cross-entropy, the optimiser is AdamW, and the best epoch is chosen by validation P-H@50.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
exclude_pairs = collect_all_pairs(ctx)
rng = np.random.default_rng(SEED)

history = {"train_loss": [], "val_ph10": [], "val_ph50": [], "val_cp_auc": []}
best = {"val_ph50": -1.0, "epoch": 0, "state": None}

print(f"{'epoch':>5}  {'loss':>7}  {'P-H@10':>7}  {'P-H@50':>7}  {'CP-AUC':>7}")
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    data = ctx.train_data
    label = data[ctx.target_edge].edge_label
    pos_ei = data[ctx.target_edge].edge_label_index[:, label == 1]
    eli, labels = sample_training_negatives(
        pos_ei, ctx.conv_idxs, NEG_K, rng, ctx.device,
        n_prot=data[ctx.src_type].num_nodes, neg_k_cp=NEG_K_CP, exclude_pairs=exclude_pairs,
    )
    logits = model(ctx, data.x_dict, data.edge_index_dict, eli)
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    model.eval()
    phk = protein_hits_at_k(model.gnn, model.predictor, ctx.val_data, ctx,
                            k_list=(10, 50), eval_embedded_only=True)
    cp_auc, _ = evaluate_cp_auc(model.gnn, model.predictor, ctx.val_data, ctx,
                                rng=np.random.default_rng(SEED + epoch), neg_k=1,
                                exclude_pairs=exclude_pairs)

    history["train_loss"].append(loss.item())
    history["val_ph10"].append(phk["P-H@10"])
    history["val_ph50"].append(phk["P-H@50"])
    history["val_cp_auc"].append(cp_auc)
    if phk["P-H@50"] > best["val_ph50"]:
        best = {"val_ph50": phk["P-H@50"], "epoch": epoch,
                "state": {k: v.detach().clone() for k, v in model.state_dict().items()}}
    if epoch % 5 == 0:
        print(f"{epoch:>5}  {loss.item():>7.4f}  {phk['P-H@10']:>7.4f}  {phk['P-H@50']:>7.4f}  {cp_auc:>7.4f}")

print(f"\nBest epoch: {best['epoch']}  (val P-H@50 = {best['val_ph50']:.4f})")

### 3.1 Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"])
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss"); axes[0].set_title("Training loss")
axes[1].plot(history["val_ph10"], label="P-H@10")
axes[1].plot(history["val_ph50"], label="P-H@50")
axes[1].plot(history["val_cp_auc"], label="CP-AUC")
axes[1].set_xlabel("Epoch"); axes[1].set_title("Validation metrics"); axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Evaluation of the best checkpoint

In [ ]:
model.load_state_dict(best["state"])
model.eval()

print(f"{'split':<6} {'P-H@1':>7} {'P-H@5':>7} {'P-H@10':>7} {'P-H@50':>7} {'CP-AUC':>7} {'CP-AP':>7}")
for name, split in [("val", ctx.val_data), ("test", ctx.test_data)]:
    phk = protein_hits_at_k(model.gnn, model.predictor, split, ctx,
                            k_list=(1, 5, 10, 50), eval_embedded_only=True)
    cp_auc, cp_ap = evaluate_cp_auc(model.gnn, model.predictor, split, ctx,
                                    rng=np.random.default_rng(SEED + 100), neg_k=1,
                                    exclude_pairs=exclude_pairs)
    print(f"{name:<6} {phk['P-H@1']:>7.4f} {phk['P-H@5']:>7.4f} {phk['P-H@10']:>7.4f} "
          f"{phk['P-H@50']:>7.4f} {cp_auc:>7.4f} {cp_ap:>7.4f}")
print(f"\nRandom P-H@50 baseline: {50 / pool:.4f}")

## 5. Running the full experiments

The reported results use the command-line scripts, which run the full schedule and write checkpoints, logs and reports to `runs/`:

```bash
python training/train.py                                     # this notebook's baseline, 200 epochs
python training/train.py --gnn-name residual_jumping_sage    # HeteroSAGE+Res&Jump
python training/train.py --no-embeddings                     # structural baseline (random features)
python training/train_seeds.py                               # 5 seeds, writes seeds_summary.json
```

The shortcut controls of §1 can be switched off one at a time with `--keep-pathways`, `--keep-catalyzed-by`, `--disjoint-train-ratio 0` and `--ec-features`, and the holdout is chosen with `--split-type` (`taxa`, `pathway`, `random`, `ath_pathway`). `training/README.md` lists all options.